# STEP 1: Setup & Overview
# Description: This script loops through each building folder, reads the annual_aggregate.csv,
# and writes its values into the corresponding annual_savings.csv using the defined mapping.

"""
Script Overview:
- Reads EPC dataset to get BUILDING_REFERENCE_NUMBER values
- For each building:
    - Opens /TOTAL/annual_aggregate.csv
    - Opens /TOTAL/annual_savings.csv
    - Transfers values using:
        Columns (1–9) -> Rows (BASELINE → BATTERY)
        Rows (energy, carbon, fixed_cost, flex_cost) -> Columns (ENERGY, CARBON, COST_FIXED, COST_FLEX)
- Saves updated annual_savings.csv back to disk
"""

In [1]:
# STEP 2: Import Libraries & Define Paths
# Description: Load required libraries and define base file paths.

import os
import pandas as pd

epc_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_with_lifetimes_2.csv"
base_models_path = "/Users/jakegehrung/Desktop/data_raw/baseline_models"

In [2]:
# STEP 3: Load EPC Dataset
# Description: Extract BUILDING_REFERENCE_NUMBER values.

epc_df = pd.read_csv(epc_path)
building_ids = epc_df["BUILDING_REFERENCE_NUMBER"].astype(str).tolist()

print(f"Total buildings found: {len(building_ids)}")

Total buildings found: 117


In [3]:
# STEP 4: Define Mapping Logic
# Description: Define mappings between aggregate columns/rows and savings rows/columns.

column_to_label = {
    "1": "BASELINE",
    "2": "WINDOWS",
    "3": "WALLS",
    "4": "ROOF",
    "5": "FAB",
    "6": "HP",
    "7": "SOLAR_THERMAL",
    "8": "PV",
    "9": "BATTERY"
}

row_to_column = {
    "energy": "ENERGY",
    "carbon": "CARBON",
    "fixed_cost": "COST_FIXED",
    "flex_cost": "COST_FLEX"
}

In [4]:
# STEP 5: Process Each Building
# Description: Loop through buildings and transfer values.

for bid in building_ids:
    total_path = os.path.join(base_models_path, bid, "TOTAL")
    agg_path = os.path.join(total_path, "annual_aggregate.csv")
    sav_path = os.path.join(total_path, "annual_savings.csv")

    # Skip if files do not exist
    if not os.path.exists(agg_path) or not os.path.exists(sav_path):
        print(f"Skipping {bid} (missing files)")
        continue

    try:
        agg_df = pd.read_csv(agg_path)
        sav_df = pd.read_csv(sav_path)

        # Ensure LABEL column exists
        if "LABEL" not in sav_df.columns:
            print(f"Skipping {bid} (missing LABEL column)")
            continue

        # Set index for easy lookup
        sav_df.set_index("LABEL", inplace=True)

        # Transfer values
        for col, label in column_to_label.items():
            if col not in agg_df.columns:
                continue

            for row, target_col in row_to_column.items():
                value = agg_df.loc[agg_df["label"] == row, col].values
                if len(value) > 0:
                    sav_df.loc[label, target_col] = value[0]

        # Reset index before saving
        sav_df.reset_index(inplace=True)

        # Save updated file
        sav_df.to_csv(sav_path, index=False)

        print(f"Processed {bid}")

    except Exception as e:
        print(f"Error processing {bid}: {e}")

Processed 1001829067
Processed 1001951858
Processed 1001829071
Processed 1234761001
Processed 1001991633
Processed 1001664929
Processed 1001829059
Processed 1001063639
Processed 1234761000
Processed 1236594950
Processed 1001664925
Processed 1001906271
Processed 1000238911
Processed 1001829057
Processed 1234760999
Processed 1002143357
Processed 1001951854
Processed 1001829069
Processed 1002313096
Processed 1002143351
Processed 1001870854
Processed 1001870864
Processed 1002143293
Processed 1002143352
Processed 1002143353
Processed 1234647955
Processed 1002313093
Processed 1001991629
Processed 1001991628
Processed 1000238920
Processed 1001829085
Processed 1001063646
Processed 1001829058
Processed 1234806523
Processed 1001664941
Processed 1236034494
Processed 1000336709
Processed 1234761002
Processed 1002143355
Processed 1001906269
Processed 1001870855
Processed 1001664944
Processed 1000336711
Processed 1001829079
Processed 1001870859
Processed 1001664928
Processed 1234806526
Processed 100

In [5]:
# STEP 6: Verification (Optional)
# Description: Check one building to confirm correct transfer.

test_id = building_ids[0]
test_path = os.path.join(base_models_path, test_id, "TOTAL", "annual_savings.csv")

df_test = pd.read_csv(test_path)
df_test.head()

,LABEL,ENERGY,CARBON,COST_FIXED,COST_FLEX,ENERGY_SAVINGS,CARBON_SAVINGS,COST_SAVINGS_FIXED,COST_SAVINGS_FLEX
0,BASELINE,2.520841e+07,1714.171908,6473.519794,4972.737329,NaN,NaN,NaN,NaN
1,WINDOWS,2.682384e+07,1824.021051,6888.361851,5294.882898,-1.615429e+06,-109.849143,-414.842057,-322.145569
2,WALLS,2.635330e+07,1792.024275,6767.526969,5204.158203,-1.144888e+06,-77.852367,-294.007176,-231.420874
3,ROOF,2.793884e+07,1899.841051,7174.693851,5520.023773,-2.730429e+06,-185.669143,-701.174057,-547.286444
4,FAB,2.520841e+07,1714.171908,6473.519794,4972.737329,NaN,NaN,NaN,NaN
